# Plot Light Curves vs MJD for Stable Stars

For each stable star in `data_MergeVisits_02_out/per_star/`, this notebook
produces a **GridSpec 6×2 figure** per star:

- **Left column (6 rows)** — `psfFlux` vs `expMidptMJD` per band (order: u g r i z y).  
  A secondary top x-axis shows the calendar date (YYYY-MM-DD).  
  The y-axis is clipped to ±N·σ_IQR around the median to make scatter vs.
  error bars immediately visible.  
  Individual error bars (`psfFluxErr`) are drawn; the Gaussian fit mean is
  shown as a horizontal dashed line and the ±σ_IQR band as a shaded region.

- **Right column (6 rows)** — histogram of `psfFlux` for the same band with a
  Gaussian fit overlaid; mean (μ) and σ from the fit are annotated.

Bands with fewer than 3 good measurements are skipped (empty subplot).

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-27
- **Last update:** 2026-06-27


## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator

from astropy.time import Time
from scipy.stats import norm
from scipy.optimize import curve_fit

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)
if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)
log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ──────────────────────────────────────────────────────────────
NB_TAG = "PlotLC_03"

# ── Input: merged LC files from notebook 02 ───────────────────────────────────
DIR_DATA_IN = "./data_MergeVisits_02_out"
DIR_PER_STAR_IN = os.path.join(DIR_DATA_IN, "per_star")

# ── Output figures ────────────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

# ── Photometric columns ───────────────────────────────────────────────────────
MJD_COL = "expMidptMJD"
FLUX_COL = "psfFlux"
FLUX_ERR_COL = "psfFluxErr"
BAND_COL = "band"

# ── Band ordering and colours (LSST ugrizy) ───────────────────────────────────
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]
BAND_COLORS = {
    "u": "#6C2DC7",  # violet
    "g": "#1CB94A",  # green
    "r": "#E8262A",  # red
    "i": "#E87E26",  # orange
    "z": "#C95C93",  # pink
    "y": "#994D00",  # brown
}

# ── Y-axis clipping for the light-curve panels ────────────────────────────────
# The y-axis is set to  median ± N_SIGMA_YLIM × σ_IQR  where
#   σ_IQR = IQR / 1.349  (robust estimator of the standard deviation)
# Increase N_SIGMA_YLIM to see more outliers; decrease to zoom in.
N_SIGMA_YLIM = 5.0

# ── Minimum number of good points to attempt a Gaussian fit ───────────────────
MIN_POINTS = 5

# ── Figure size (width, height) in inches ─────────────────────────────────────
FIG_WIDTH = 16
FIG_HEIGHT = 22

## 4. Helper functions

In [ ]:
# ── MJD → datetime (for the secondary top x-axis) ───────────────────────────
def mjd_to_datetime(mjd_array):
    """Convert an array of MJD floats to numpy datetime64 objects."""
    return Time(mjd_array, format="mjd").to_datetime()


# ── Robust σ from IQR ────────────────────────────────────────────────────────
def sigma_iqr(values):
    """Robust standard-deviation estimate via the inter-quartile range.
    σ_IQR = IQR / 1.3489795
    """
    q75, q25 = np.nanpercentile(values, [75, 25])
    return (q75 - q25) / 1.3489795


# ── Gaussian model for curve_fit ─────────────────────────────────────────────
def gaussian(x, mu, sigma, amplitude):
    return amplitude * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


# ── Fit a Gaussian to a histogram ────────────────────────────────────────────
def fit_gaussian_to_flux(flux_values, n_bins=30):
    """Fit a Gaussian to a histogram of *flux_values*.

    Returns
    -------
    (mu, sigma, amplitude, bin_centers, hist_counts) or
    (None, None, None, bin_centers, hist_counts) on failure.
    """
    counts, bin_edges = np.histogram(flux_values, bins=n_bins)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    mu0 = np.median(flux_values)
    sigma0 = sigma_iqr(flux_values)
    amp0 = counts.max()

    if sigma0 <= 0 or amp0 <= 0:
        return None, None, None, bin_centers, counts

    try:
        popt, _ = curve_fit(
            gaussian,
            bin_centers,
            counts,
            p0=[mu0, sigma0, amp0],
            maxfev=5000,
        )
        mu_fit, sigma_fit, amp_fit = popt
        # Reject unphysical fits
        if sigma_fit <= 0 or abs(mu_fit - mu0) > 5 * sigma0:
            return None, None, None, bin_centers, counts
        return abs(mu_fit), abs(sigma_fit), abs(amp_fit), bin_centers, counts
    except RuntimeError:
        return None, None, None, bin_centers, counts


# ── savefig: PDF + PNG ───────────────────────────────────────────────────────
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)

## 5. Core plotting function: one star → one 6×2 GridSpec figure

In [ ]:
def plot_star_lc(df_star: pd.DataFrame, simbad_id: str, file_stem: str) -> None:
    """Build and save the 6×2 GridSpec figure for one stable star.

    Parameters
    ----------
    df_star    : merged LC DataFrame for this star.
    simbad_id  : human-readable identifier shown in the figure title.
    file_stem  : output filename stem (no extension, no directory).
    """
    # Drop rows with NaN flux or MJD
    df = df_star.dropna(subset=[MJD_COL, FLUX_COL]).copy()

    # ── Figure & GridSpec ────────────────────────────────────────────────────
    fig = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
    fig.suptitle(
        f"{simbad_id}\n" f"psfFlux light curves per band   |   y-axis: median ± {N_SIGMA_YLIM:.0f}·σ_IQR",
        fontsize=13,
        fontweight="bold",
        y=0.995,
    )

    # 6 rows × 2 columns; left panels wider than right
    gs = gridspec.GridSpec(
        6,
        2,
        figure=fig,
        width_ratios=[3, 1],
        hspace=0.55,
        wspace=0.28,
    )

    for row_idx, band in enumerate(BAND_ORDER):
        color = BAND_COLORS.get(band, "steelblue")

        ax_lc = fig.add_subplot(gs[row_idx, 0])  # left: light curve
        ax_hist = fig.add_subplot(gs[row_idx, 1])  # right: histogram

        # Select this band, require valid flux and error
        mask = (df[BAND_COL] == band) & df[MJD_COL].notna() & df[FLUX_COL].notna()
        # Also exclude flagged rows if psfFluxErr is 0 or NaN
        if FLUX_ERR_COL in df.columns:
            mask &= df[FLUX_ERR_COL].notna() & (df[FLUX_ERR_COL] > 0)

        grp = df[mask].sort_values(MJD_COL)
        n_pts = len(grp)

        # ── Empty band subplot ───────────────────────────────────────────────
        if n_pts < MIN_POINTS:
            for ax in (ax_lc, ax_hist):
                ax.set_visible(True)
                ax.text(
                    0.5,
                    0.5,
                    f"{band}  —  {n_pts} point(s)",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    color="grey",
                    fontsize=10,
                )
                ax.set_xticks([])
                ax.set_yticks([])
            continue

        mjd = grp[MJD_COL].values
        flux = grp[FLUX_COL].values
        err = grp[FLUX_ERR_COL].values if FLUX_ERR_COL in grp.columns else np.zeros_like(flux)

        # ── Robust statistics ────────────────────────────────────────────────
        med = np.nanmedian(flux)
        sig_iqr = sigma_iqr(flux)
        if sig_iqr == 0:
            sig_iqr = np.nanstd(flux) or 1.0

        # ── Gaussian fit (used for histogram annotation) ─────────────────────
        mu_fit, sigma_fit, amp_fit, bin_centers, hist_counts = fit_gaussian_to_flux(flux)

        # ── LEFT panel: light curve ──────────────────────────────────────────
        ax_lc.errorbar(
            mjd,
            flux,
            yerr=err,
            fmt="o",
            ms=4,
            color=color,
            ecolor=color,
            alpha=0.8,
            elinewidth=0.8,
            capsize=2,
            label=f"{band}  (N={n_pts})",
            zorder=3,
        )

        # Median line
        ax_lc.axhline(med, color=color, lw=1.4, ls="--", alpha=0.9, zorder=2)

        # ±σ_IQR shaded band
        ax_lc.axhspan(
            med - sig_iqr,
            med + sig_iqr,
            color=color,
            alpha=0.10,
            zorder=1,
        )

        # y-axis limits: median ± N_SIGMA_YLIM × σ_IQR
        ylim_lo = med - N_SIGMA_YLIM * sig_iqr
        ylim_hi = med + N_SIGMA_YLIM * sig_iqr
        # Safety: never collapse to zero range
        if ylim_hi - ylim_lo < 1e-10 * abs(med) + 1.0:
            ylim_lo = med - 1.0
            ylim_hi = med + 1.0
        ax_lc.set_ylim(ylim_lo, ylim_hi)

        # Annotation: σ_IQR / median  (relative scatter in %)
        rel_scatter = 100.0 * sig_iqr / abs(med) if abs(med) > 0 else np.nan
        ax_lc.text(
            0.02,
            0.96,
            f"σ_IQR = {sig_iqr:.1f} nJy   ({rel_scatter:.2f}%)",
            transform=ax_lc.transAxes,
            fontsize=8,
            va="top",
            ha="left",
            color=color,
        )

        ax_lc.set_ylabel(f"{band}  psfFlux [nJy]", fontsize=9, color=color)
        ax_lc.tick_params(axis="y", labelsize=8)
        ax_lc.yaxis.set_minor_locator(AutoMinorLocator())
        ax_lc.tick_params(which="minor", length=2)

        # x-axis: MJD ticks on the bottom, date labels on top (first row only
        # gets the secondary axis; all rows share the same tick style)
        ax_lc.tick_params(axis="x", labelsize=7)

        # Secondary top x-axis with calendar dates (all rows)
        ax_top = ax_lc.twiny()
        ax_top.set_xlim(ax_lc.get_xlim())

        # Choose a handful of tick positions evenly across the MJD range
        mjd_min, mjd_max = mjd.min(), mjd.max()
        mjd_range = mjd_max - mjd_min
        if mjd_range < 1:
            # All observations within one day — show hh:mm instead
            n_ticks = 4
        else:
            n_ticks = min(5, max(2, int(mjd_range // 30) + 2))
        tick_mjd = np.linspace(mjd_min, mjd_max, n_ticks)
        tick_dates = Time(tick_mjd, format="mjd").to_value("iso", subfmt="date")

        ax_top.set_xticks(tick_mjd)
        ax_top.set_xticklabels(tick_dates, fontsize=7, rotation=20, ha="left")
        ax_top.tick_params(length=3)

        if row_idx == len(BAND_ORDER) - 1:
            ax_lc.set_xlabel("expMidptMJD", fontsize=9)

        ax_lc.legend(loc="upper right", fontsize=8, framealpha=0.6)
        ax_lc.grid(True, lw=0.4, alpha=0.4)

        # ── RIGHT panel: histogram + Gaussian fit ────────────────────────────
        ax_hist.bar(
            bin_centers,
            hist_counts,
            width=(bin_centers[1] - bin_centers[0]) * 0.9,
            color=color,
            alpha=0.55,
            edgecolor="none",
            label="data",
        )

        if mu_fit is not None:
            x_fit = np.linspace(bin_centers[0], bin_centers[-1], 300)
            y_fit = gaussian(x_fit, mu_fit, sigma_fit, amp_fit)
            ax_hist.plot(x_fit, y_fit, color="k", lw=1.5, label="Gauss fit")
            ax_hist.axvline(mu_fit, color="k", lw=1.2, ls="--")
            ax_hist.axvline(mu_fit - sigma_fit, color="grey", lw=0.9, ls=":")
            ax_hist.axvline(mu_fit + sigma_fit, color="grey", lw=0.9, ls=":")
            ax_hist.text(
                0.97,
                0.96,
                f"μ = {mu_fit:.1f}\nσ = {sigma_fit:.1f}",
                transform=ax_hist.transAxes,
                fontsize=8,
                va="top",
                ha="right",
                color="k",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7),
            )
        else:
            ax_hist.text(
                0.97,
                0.96,
                "fit failed",
                transform=ax_hist.transAxes,
                fontsize=8,
                va="top",
                ha="right",
                color="red",
            )

        ax_hist.set_xlabel("psfFlux [nJy]", fontsize=8)
        ax_hist.set_ylabel("counts", fontsize=8)
        ax_hist.tick_params(labelsize=8)
        ax_hist.legend(fontsize=7, loc="upper left")
        ax_hist.set_title(f"{band} band", fontsize=9, color=color)
        ax_hist.grid(True, lw=0.3, alpha=0.35)

    savefig(fig, file_stem)
    plt.close(fig)
    gc.collect()

## 6. Discover per-star files and run the plotting loop

In [ ]:
lc_files = sorted(f for f in os.listdir(DIR_PER_STAR_IN) if f.endswith("_lc_mjd.csv"))
log.info("Found %d per-star LC files.", len(lc_files))
lc_files

In [ ]:
n_ok = 0
n_err = 0

for fname in lc_files:
    src_path = os.path.join(DIR_PER_STAR_IN, fname)

    try:
        df_star = pd.read_csv(src_path)
    except Exception as exc:
        log.error("  ERROR reading %s: %s", fname, exc)
        n_err += 1
        continue

    # simbad_id from the first valid row
    simbad_id = (
        df_star["simbad_id"].iloc[0]
        if "simbad_id" in df_star.columns and len(df_star) > 0
        else fname.replace("_lc_mjd.csv", "")
    )

    # Output figure stem: strip the suffix
    file_stem = fname.replace("_lc_mjd.csv", "") + "_LC_MJD"

    log.info("Plotting: %s  (%d rows)", simbad_id, len(df_star))

    try:
        plot_star_lc(df_star, simbad_id, file_stem)
        n_ok += 1
    except Exception as exc:
        log.error("  ERROR plotting %s: %s", simbad_id, exc)
        plt.close("all")
        n_err += 1

log.info("Done — %d figures saved, %d errors.", n_ok, n_err)

## 7. Quick inline preview of one figure

In [ ]:
# Display the first saved PNG inline for a quick check
from IPython.display import Image, display

saved_pngs = sorted(os.path.join(DIR_FIGS, f) for f in os.listdir(DIR_FIGS) if f.endswith(".png"))

if saved_pngs:
    log.info("Previewing: %s", saved_pngs[0])
    display(Image(filename=saved_pngs[0], width=900))
else:
    log.warning("No PNG figure found in %s", DIR_FIGS)

## 8. Summary table: scatter vs. median photon-noise per star and band

In [ ]:
# Load the global merged file to build the summary table
global_csv = os.path.join(DIR_DATA_IN, "all_stars_lightcurves_mjd.csv")
df_all = pd.read_csv(global_csv)
df_all = df_all.dropna(subset=[MJD_COL, FLUX_COL])
if FLUX_ERR_COL in df_all.columns:
    df_all = df_all[df_all[FLUX_ERR_COL] > 0]

rows = []
for (sid, band), grp in df_all.groupby(["simbad_id", BAND_COL]):
    flux = grp[FLUX_COL].values
    err = grp[FLUX_ERR_COL].values if FLUX_ERR_COL in grp.columns else np.full_like(flux, np.nan)
    if len(flux) < MIN_POINTS:
        continue
    med = np.nanmedian(flux)
    sig_iqr_ = sigma_iqr(flux)
    med_err = np.nanmedian(err)
    rows.append(
        {
            "simbad_id": sid,
            "band": band,
            "N": len(flux),
            "median_flux_nJy": round(med, 2),
            "sigma_IQR_nJy": round(sig_iqr_, 2),
            "median_err_nJy": round(med_err, 2),
            "scatter/err": round(sig_iqr_ / med_err, 3) if med_err > 0 else np.nan,
            "rel_scatter_%": round(100 * sig_iqr_ / abs(med), 4) if abs(med) > 0 else np.nan,
        }
    )

df_summary = pd.DataFrame(rows).sort_values(["simbad_id", "band"])

# Highlight rows where scatter exceeds photon noise by more than 2×
df_summary["excess_scatter"] = df_summary["scatter/err"] > 2.0

print(f"Summary table — {len(df_summary)} (star × band) combinations")
df_summary

In [ ]:
# Save summary table
summary_out = os.path.join(DIR_FIGS, "scatter_vs_noise_summary.csv")
df_summary.to_csv(summary_out, index=False)
log.info("Scatter summary saved: %s", summary_out)

# Stars with excess scatter in at least one band
excess = df_summary[df_summary["excess_scatter"]]
if len(excess):
    print(f"\n{len(excess)} (star × band) combinations with scatter/err > 2:")
    print(
        excess[["simbad_id", "band", "N", "sigma_IQR_nJy", "median_err_nJy", "scatter/err"]].to_string(
            index=False
        )
    )
else:
    print("No (star × band) combination with scatter/err > 2 found.")